In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
hospital_df = spark.read.table(
    "hospital_appointments.bronze.batch1_raw_data"
)

In [0]:
from pyspark.sql.functions import (
    col, trim, lower, coalesce, try_to_date
)

# 1. appointment_id
condition1 = (
    col("appointment_id").isNull() |
    (trim(col("appointment_id")) == "")
)

# 2. patient_id
condition2 = (
    col("patient_id").isNull() |
    (trim(col("patient_id")) == "")
)

# 3. patient_name
condition3 = (
    col("patient_name").isNull() |
    (trim(col("patient_name")) == "")
)

# 4. patient_age
condition4 = (
    col("patient_age").isNull() |
    (col("patient_age") <= 0) |
    (col("patient_age") > 120)
)

# 5. patient_gender
condition5 = (
    col("patient_gender").isNull() |
    (~lower(trim(col("patient_gender"))).isin(
        "m","f"
    ))
)

# 6. hospital_city
condition6 = (
    col("hospital_city").isNull() |
    (trim(col("hospital_city")) == "")
)

# 7. doctor_id
condition7 = (
    col("doctor_id").isNull() |
    (trim(col("doctor_id")) == "")
)

# 8. doctor_name
condition8 = (
    col("doctor_name").isNull() |
    (trim(col("doctor_name")) == "")
)

# 9. department
condition9 = (
    col("department").isNull() |
    (trim(col("department")) == "")
)

# 10. appointment_date
parsed_date = coalesce(
    try_to_date(col("appointment_date"), "yyyy-MM-dd"),
    try_to_date(col("appointment_date"), "dd-MM-yyyy"),
     try_to_date(col("appointment_date"), "MM/dd/yyyy"),
    try_to_date(col("appointment_date"), "M/d/yyyy")
)

condition10 = (
    col("appointment_date").isNull() |
    parsed_date.isNull()
)

# 11. consultation_fee
condition11 = (
    col("consultation_fee").isNull() |
    (col("consultation_fee") <= 0)
)

# 12. test_charge
condition12 = (
    col("test_charge").isNull() |
    (col("test_charge") < 0)
)

# 13. appointment_status
condition13 = (
    col("appointment_status").isNull() |
    (~lower(trim(col("appointment_status"))).isin(
        "completed",
        "scheduled",
        "cancelled"
      
    ))
)


In [0]:
invalid_conditions=condition1|condition2|condition3|condition4|condition5|condition6|condition7|condition8|condition9|condition10|condition11|condition12|condition13

In [0]:
invalid_records_df=(
    hospital_df
    .filter(invalid_conditions)
)

In [0]:
display(invalid_records_df)

In [0]:
invalid_records=invalid_records_df.count()
print("Number of invalid records: ",invalid_records)

In [0]:
invalid_records_df= (
   invalid_records_df
    .withColumn(
        "rejection_reason",
        concat_ws(
            ", ",
            when(condition1, lit("Missing Appointment ID")),
            when(condition2, lit("Missing Patient ID")),
            when(condition3, lit("Missing Patient Name")),
            when(condition4, lit("Invalid Patient Age")),
            when(condition5, lit("Invalid Patient Gender")),
            when(condition6, lit("Missing hospital City")),
            when(condition7, lit("Missing Doctor ID")),
            when(condition8, lit("Missing Doctor Name")),
            when(condition9, lit("Missing Department")),
            when(condition10, lit("Invalid Appointment Date")),
            when(condition11, lit("Invalid Consultation Fee")),
            when(condition12, lit("Invalid Test Charge")),
            when(condition13, lit("Invalid Appointment Status"))
        )
    )
)

In [0]:
display(invalid_records_df)

In [0]:
invalid_records_df.write.mode("overwrite").format("delta").saveAsTable("hospital_appointments.bronze.batch1_invalid_records")


In [0]:
display(spark.table("hospital_appointments.bronze.batch1_invalid_records"))

In [0]:
valid_records_df=(
    hospital_df
    .filter(~invalid_conditions)
)
display(valid_records_df)


In [0]:
valid_records=valid_records_df.count()
print("Number of valid records: ",valid_records)

In [0]:
#checking duplicates records based on appointment_id
duplicate_records_df = valid_records_df.groupBy("appointment_id") \
    .count() \
    .filter("count > 1")

# Drop duplicate records
valid_records_df = valid_records_df.dropDuplicates(["appointment_id"])




In [0]:
# Count the number of duplicate records
duplicate_records = duplicate_records_df.count()

# Print the number of duplicate records
print(f"Number of duplicate records: ",duplicate_records)

In [0]:
duplicate_records_df.write.mode("overwrite").format("delta").saveAsTable("hospital_appointments.bronze.batch1_duplicate_records")
display(spark.table("hospital_appointments.bronze.batch1_duplicate_records"))

In [0]:
from pyspark.sql.functions import *

valid_records_df = (
    valid_records_df

    # Standardize patient name
    .withColumn(
        "patient_name",
        initcap(trim(col("patient_name")))
    )

    # Standardize city
    .withColumn(
        "hospital_city",
        initcap(trim(col("hospital_city")))
    )

    # Standardize doctor name
    .withColumn(
        "doctor_name",
        initcap(trim(col("doctor_name")))
    )

    # Standardize department (preserve ENT)
    .withColumn(
        "department",
        when(
            upper(trim(col("department"))) == "ENT",
            lit("ENT")
        ).otherwise(
            initcap(trim(col("department")))
        )
    )

    # Standardize status
    .withColumn(
        "appointment_status",
        initcap(trim(col("appointment_status")))
    )

    # Convert appointment date
    .withColumn(
        "appointment_date",
        parsed_date
    )

    # Cast consultation fee
    .withColumn(
        "consultation_fee",
        col("consultation_fee").cast(DoubleType())
    )

    # Cast test charge
    .withColumn(
        "test_charge",
        col("test_charge").cast(DoubleType())
    )

    # Calculate total appointment amount
    .withColumn(
        "appointment_amount",
        col("consultation_fee") + col("test_charge")
    )
)

In [0]:
display(valid_records_df)

In [0]:
valid_records_df.printSchema()

In [0]:
silver_records=valid_records_df.count()
print("Number of silver records: ",silver_records)

In [0]:
valid_records_df.write.mode("overwrite").format("delta").saveAsTable("hospital_appointments.silver.silver_data")
display(spark.table("hospital_appointments.silver.silver_data"))